# Reflection — 최선의 답변을 위한 반복

**Reflection(반성)** 은 LLM 이 자신이 만든 결과물을 스스로(또는 다른 LLM 이) 평가·비평하고, 그 피드백을 반영해 결과를 개선하는 패턴이다. "생성 → 비평 → 개선" 을 반복해 품질을 끌어올린다.

이 노트북에서 다루는 것:
1. **기본 Reflection** — 생성 LLM ↔ 비평 LLM 을 번갈아 호출 (에세이 작성 예제)
2. **그래프로 구현** — `generate` 노드 ↔ `reflect` 노드 루프
3. **Reflexion** — 자기반성 + 웹검색으로 근거를 보강하며 답변을 수정하는 고급 패턴

> 실제 LLM 을 호출하므로 API 키가 필요하다.

## 환경 변수 준비

프로젝트 루트의 `.env` 에 키를 넣고 `load_dotenv()` 로 불러온다.

```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...     # 웹검색이 필요한 노트북
ANTHROPIC_API_KEY=sk-ant-... # Claude 를 쓰는 노트북
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료:", "OPENAI_API_KEY")

## 1. 기본 Reflection — 생성과 비평을 번갈아

두 개의 체인을 만든다.
- **generate** : 에세이를 작성하는 도우미
- **reflect** : 그 에세이를 채점·비평하는 교사

[basics 복습] `ChatPromptTemplate` 로 system 역할(페르소나)을 주고, `MessagesPlaceholder` 로 대화 메시지가 들어갈 자리를 비워둔다.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

# 생성 체인: 에세이 작성 도우미
generate_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 5단락 에세이를 훌륭하게 작성하는 에세이 도우미입니다."
     "사용자의 요청에 따라 최고의 에세이를 작성하세요."
     "사용자가 피드백을 제공할 경우, 이전 시도에서 개선된 수정본을 작성해 응답하세요."),
    MessagesPlaceholder(variable_name="messages"),
])
generate = generate_prompt | llm

에세이를 한 번 생성해본다. [basics 복습] `stream` 으로 토큰을 실시간 출력하며 누적한다.

In [ ]:
request = HumanMessage(content="AI Agent의 중요성에 대한 에세이를 작성해주세요.")

essay = ""
for chunk in generate.stream({"messages": [request]}):
    print(chunk.content, end="")
    essay += chunk.content

이제 **비평 체인** 을 만들어 위 에세이를 채점한다.

In [ ]:
reflection_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 에세이를 채점하는 교사입니다. 사용자가 제출한 에세이에 대한 비평과 개선 사항을 작성하세요."
     "글의 길이, 깊이, 문체 등을 포함해 구체적인 개선 요청을 제공하세요."),
    MessagesPlaceholder(variable_name="messages"),
])
reflect = reflection_prompt | llm

reflection = ""
for chunk in reflect.stream({"messages": [request, HumanMessage(content=essay)]}):
    print(chunk.content, end="")
    reflection += chunk.content

비평을 피드백으로 넘겨 **개선된 에세이** 를 다시 생성한다. 이때 직전 에세이는 `AIMessage`(내가 쓴 답), 비평은 `HumanMessage`(피드백) 로 넣는 게 핵심이다.

In [ ]:
for chunk in generate.stream(
    {"messages": [request, AIMessage(content=essay), HumanMessage(content=reflection)]}
):
    print(chunk.content, end="")

## 2. 그래프로 Reflection 구현하기

위 "생성 → 비평 → 생성 → …" 흐름을 그래프 루프로 만든다.

```
START → generate ──(메시지 6개 초과?)──▶ END
          ▲                              
          └──────── reflect ◀────────────┘ (아니면 비평 후 다시 생성)
```

[basics 복습] State 는 메시지 누적(`add_messages`), 분기는 조건부 엣지로 구현한다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    messages: Annotated[list, add_messages]

**generate 노드** — 현재까지의 메시지로 에세이(또는 수정본)를 생성.

In [ ]:
def generation_node(state: State) -> State:
    return {"messages": [generate.invoke(state["messages"])]}

**reflect 노드** — 핵심 트릭: 비평 LLM 입장에서는 "내가 생성한 에세이"(`AIMessage`)를 "사용자가 제출한 글"(`HumanMessage`)처럼 보여줘야 채점한다. 그래서 역할을 뒤집어(`ai`↔`human`) 변환한다.

[basics 복습] 메시지의 `.type`(role)으로 분기해 타입을 바꿔치기하는 패턴이다.

In [ ]:
def reflection_node(state: State) -> State:
    # 비평 LLM 에 넘길 때 역할을 뒤집는다 (ai 가 쓴 글 → human 이 제출한 글로)
    cls_map = {"ai": HumanMessage, "human": AIMessage}
    # 첫 메시지(원 요청)는 그대로 두고, 이후 메시지들의 역할만 반전
    translated = [state["messages"][0]] + [
        cls_map[msg.type](content=msg.content) for msg in state["messages"][1:]
    ]
    res = reflect.invoke(translated)
    # 비평 결과를 (생성 노드 입장에서) 사용자 피드백으로 되돌린다
    return {"messages": [HumanMessage(content=res.content)]}

**그래프 조립** — generate 뒤에서 메시지가 6개를 넘으면 종료, 아니면 reflect 로.

In [ ]:
def should_continue(state: State):
    if len(state["messages"]) > 6:   # 원 요청 + (생성/비평) 3쌍 ≈ 충분한 반복
        return END
    return "reflect"

graph_builder = StateGraph(State)
graph_builder.add_node("generate", generation_node)
graph_builder.add_node("reflect", reflection_node)
graph_builder.add_edge(START, "generate")
graph_builder.add_conditional_edges("generate", should_continue)
graph_builder.add_edge("reflect", "generate")   # 비평 후 다시 생성 (루프)

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

실행 — 생성/비평이 번갈아 일어나며 에세이가 개선된다.

In [ ]:
config = {"configurable": {"thread_id": "1"}}

for event in graph.stream(
    {"messages": [HumanMessage(content="AI Agent의 중요성에 대한 에세이를 작성해주세요.")]},
    config,
):
    for node, value in event.items():
        print(f"\n===== [{node}] =====")
        print(value["messages"][-1].content[:300], "...")

## 3. Reflexion — 자기반성 + 웹검색으로 근거 보강

Reflexion 은 기본 Reflection 을 확장한 패턴이다 ([논문](https://arxiv.org/abs/2303.11366)):
1. **자기반성하는 Actor** — 답변하면서 "무엇이 부족/불필요한지" 스스로 비평
2. **외부 평가/근거** — 부족한 부분을 메우기 위해 웹검색(Tavily) 수행
3. **수정** — 검색 결과를 근거로 답변을 개선 (인용 포함)

여기서는 **구조화 출력을 도구로 강제**(`tool_choice`)해서, LLM 이 "답변 + 자기비평 + 검색쿼리" 를 한 번에 정해진 형식으로 내도록 한다.

In [ ]:
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY 필요"

from langchain_community.tools.tavily_search import TavilySearchResults
tavily_tool = TavilySearchResults(max_results=5)

### Step 1. 데이터 스키마 정의
[basics 복습] Pydantic `BaseModel` 로 출력 형식을 정의한다. `Reflection`(자기비평) → `AnswerQuestion`(답변+비평+검색쿼리) 순으로 중첩한다.

In [ ]:
from pydantic import BaseModel, Field

class Reflection(BaseModel):
    missing: str = Field(description="누락되거나 부족한 부분에 대한 비평")
    superfluous: str = Field(description="불필요한 부분에 대한 비평")

class AnswerQuestion(BaseModel):
    """질문에 답하고, 자기반성한 뒤, 답변 개선을 위한 검색 쿼리를 제시한다."""
    answer: str = Field(description="질문에 대한 10문장 이내의 자세한 답변")
    search_queries: list[str] = Field(
        description="비평을 해결하기 위한 1~3개의 웹 검색 쿼리"
    )
    reflection: Reflection = Field(description="답변에 대한 자기반성")

### Step 2. 초기 답변기 (Initial responder)

[basics 복습] `bind_tools(..., tool_choice="any")` 로 **반드시 그 스키마 형태로 출력**하게 강제한다. (스키마를 도구로 사용하는 구조화 출력 방식)

In [ ]:
import datetime
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

actor_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """당신은 전문 연구자입니다.
     1. {first_instruction}
     2. 생성한 답변을 되돌아보고 개선할 수 있도록 비판하세요.
     3. 답변의 질을 높이기 위해 추가 조사할 웹 검색 쿼리를 추천하세요."""),
    MessagesPlaceholder(variable_name="messages"),
    ("user", "사용자 원래 질문과 지금까지의 행동을 되돌아보세요."),
])

initial_answer_chain = actor_prompt.partial(
    first_instruction="질문에 대한 10문장 이내의 자세한 답변을 제공해주세요.",
) | llm.bind_tools(tools=[AnswerQuestion], tool_choice="any")

**Responder** — 체인을 감싸 state 의 messages 로 호출하는 헬퍼.

In [ ]:
class Responder:
    def __init__(self, runnable):
        self.runnable = runnable

    def respond(self, state: dict):
        response = self.runnable.invoke({"messages": state["messages"]})
        return {"messages": response}

first_responder = Responder(runnable=initial_answer_chain)

example_question = "AI Agent가 무엇인가요?"
initial = first_responder.respond({"messages": [HumanMessage(content=example_question)]})
# 도구 호출 형태로 구조화된 출력 확인
initial["messages"].tool_calls[0]["args"]

### Step 3. 수정 단계 (Revision)

`AnswerQuestion` 을 상속해 `references`(인용 출처) 를 추가한 `ReviseAnswer` 스키마를 만든다.

In [ ]:
class ReviseAnswer(AnswerQuestion):
    """이전 답변을 수정한다. 답변/반성/인용출처/검색쿼리를 포함한다."""
    references: list[str] = Field(description="수정된 답변에 사용된 인용 출처")

revise_instructions = """이전 답변을 새로운 정보를 바탕으로 수정하세요.
- 이전 비평을 활용해 중요한 정보를 추가하고, 숫자 인용 표시를 포함하세요.
- 답변 하단에 '참고문헌' 섹션을 추가하세요.
- 불필요한 정보는 제거하고 최종 답변은 250자를 넘지 않게 하세요.
"""

revision_chain = actor_prompt.partial(
    first_instruction=revise_instructions,
) | llm.bind_tools(tools=[ReviseAnswer], tool_choice="any")
revisor = Responder(runnable=revision_chain)

### Step 4. 웹검색 도구 노드

[basics 복습] `StructuredTool.from_function` 으로 검색 함수를 도구화하고 `ToolNode` 로 묶는다. 도구 이름을 스키마 이름(`AnswerQuestion`/`ReviseAnswer`)과 맞춰, 해당 도구 호출이 나오면 검색이 실행되게 한다.

In [ ]:
from langchain_core.tools import StructuredTool
from langgraph.prebuilt import ToolNode

def run_queries(search_queries: list[str], **kwargs):
    """생성된 검색 쿼리들을 실행한다."""
    return tavily_tool.batch([{"query": q} for q in search_queries])

tool_node = ToolNode([
    StructuredTool.from_function(run_queries, name=AnswerQuestion.__name__),
    StructuredTool.from_function(run_queries, name=ReviseAnswer.__name__),
])

### Step 5. 그래프 생성

```
START → draft → execute_tools → revise ──(반복 횟수 초과?)──▶ END
                      ▲                          │
                      └──────────────────────────┘ (아니면 다시 검색→수정)
```

In [ ]:
MAX_ITERATIONS = 5

class ReflexionState(TypedDict):
    messages: Annotated[list, add_messages]

rb = StateGraph(ReflexionState)
rb.add_node("draft", first_responder.respond)
rb.add_node("execute_tools", tool_node)
rb.add_node("revise", revisor.respond)
rb.add_edge("draft", "execute_tools")
rb.add_edge("execute_tools", "revise")

def _get_num_iterations(messages):
    i = 0
    for m in messages[::-1]:
        if m.type not in {"tool", "ai"}:
            break
        i += 1
    return i

def event_loop(state):
    if _get_num_iterations(state["messages"]) > MAX_ITERATIONS:
        return END
    return "execute_tools"

rb.add_conditional_edges("revise", event_loop, ["execute_tools", END])
rb.add_edge(START, "draft")
reflexion_graph = rb.compile()

In [ ]:
try:
    display(Image(reflexion_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(reflexion_graph.get_graph().draw_mermaid())

실행 — draft(초안) → 검색 → revise(수정) 가 반복되며 근거 있는 답변으로 발전한다.

In [ ]:
events = reflexion_graph.stream(
    {"messages": [HumanMessage(content="AI Agent가 무엇인가요?")]},
    stream_mode="values",
)
for i, step in enumerate(events):
    print(f"\n===== Step {i} =====")
    step["messages"][-1].pretty_print()

## 정리

- **Reflection** = 생성 ↔ 비평 루프로 결과 품질을 점진 개선
- 그래프로: `generate` ↔ `reflect` 노드 + 반복 제한 조건부 엣지
- 비평 노드에서 **메시지 역할(ai↔human)을 뒤집는** 트릭이 핵심
- **Reflexion** = 자기반성 + 웹검색 근거 보강 + 인용 포함 수정 (구조화 출력을 도구로 강제)

다음: 계획을 세우고 실행·재계획을 반복하는 **Plan & Execute**.